### SCD Type 2 on Customer Table

In [0]:
from pyspark.sql.functions import *

#  Source
df = spark.read.format("delta").table("bankingdata.silver.customer")

In [0]:
df.columns

### Hash Column at Source

In [0]:
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("CustomerID").cast("string"),
        col("FirstName"),
        col("LastName"),
        col("City"),
        col("Email"),
        col("Phone")
    ))
)


### Target Table


In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankingdata.gold.dim_customer
(
    CustomerID INT,
    FirstName STRING,
    LastName STRING,
    City STRING,
    Email STRING,
    Phone STRING,
    ModifiedDate TIMESTAMP,

    HASHVALUE STRING,

    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING,

    IS_ACTIVE INT
)

In [0]:
# Target table
from delta.tables import DeltaTable
table_name = "bankingdata.gold.dim_customer"
df_tgt = DeltaTable.forName(spark, table_name)

### Detect New or Changed Records

In [0]:
df_new = (
    df_hash.alias("src")
    .join(
        df_tgt.toDF().alias("tgt"),
        (col("src.CustomerID") == col("tgt.CustomerID")) &
        (col("src.src_hash") == col("tgt.HASHVALUE")),
        "left_anti"
    )
)

### Identify Changed Records

In [0]:
LatestRecord = (
    df_new.alias("new")
    .join(
        df_tgt.toDF().alias("old"),
        col("new.CustomerID") == col("old.CustomerID"),
        "inner"
    )
    .where(
        (col("new.src_hash") != col("old.HASHVALUE")) &
        (col("old.IS_ACTIVE") == 1)
    )
    .select(
        col("new.CustomerID").alias("MergeKey"),
        col("new.*")
    )
)

### Prepare New Records


In [0]:
from pyspark.sql.functions import lit

LatestRecord1 = (
    df_new
    .select(
        lit(None).alias("MergeKey"),
        "*"
    )
)

### Combine Changed and New Records

In [0]:
updates = LatestRecord.union(LatestRecord1)

### Apply SCD Type-2 Using Delta MERGE

In [0]:
(
    df_tgt.alias("tgt")
    .merge(
        updates.alias("src"),
        "tgt.CustomerID = src.MergeKey AND tgt.IS_ACTIVE = 1"
    )
    .whenMatchedUpdate(
        condition="tgt.IS_ACTIVE = 1 AND tgt.HASHVALUE != src.src_hash",
        set={
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("Databricks-Updated"),
            "IS_ACTIVE": lit(0)
        }
    )
    .whenNotMatchedInsert(
        values={
            "CustomerID": "src.CustomerID",
            "FirstName": "src.FirstName",
            "LastName": "src.LastName",
            "City": "src.City",
            "Email": "src.Email",
            "Phone": "src.Phone",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": to_timestamp(lit("9999-01-01 00:00:00")),
            "UPDATEDBY": lit("databricks"),
            "IS_ACTIVE": lit(1)
        }
    )
    .execute()
)

In [0]:
%sql
select * from bankingdata.gold.dim_customer